In [1]:
import os

from utils.general import print_gpu_info

os.environ["CUDA_VISIBLE_DEVICES"] = "2,3"
print_gpu_info()

Available GPUs: 2
GPU 0: NVIDIA A100-SXM4-80GB | Available: 78.84 GB | Total: 79.25 GB
GPU 1: NVIDIA A100-SXM4-80GB | Available: 78.84 GB | Total: 79.25 GB


In [2]:
from llm_attribution.LLMAnalyzer import LLMAnalyzer
from llm_attribution.utils_attribution import AttributionMethod
from utils.custom_chat_template import custom_apply_chat_template


model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
analyzer = LLMAnalyzer(model_id=model_id, device="auto")
messages = []
user_prompt_decision = """I am feeling nervous about my midterm tomorrow. I fear that
Choose the most plausible answer, respond only with the answer and the description:
A) the professor will delay the midterm.
B) I will doodle on my exam and receive points for it.
C) my grandpa has diabetes.
D) I will fail.
"""
messages.append({"role": "user", "content": user_prompt_decision})
decision_prompt = custom_apply_chat_template(messages)
generated_output = analyzer.generate_output(decision_prompt)
print(f"Generated output Check: {generated_output}")
method_params = {
    AttributionMethod.LIME: {
        "n_samples": 500,
        # "baselines": analyzer.tokenizer.pad_token_id,
        "perturbations_per_eval": 500,
    },
    AttributionMethod.SHAPLEY_VALUE_SAMPLING: {
        "n_samples": 100,
        # "baselines": analyzer.tokenizer.pad_token_id,
        "perturbations_per_eval": 100,
    },
    AttributionMethod.LIG: {
        "n_steps": 60,
    },
}

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Padding token added.
Model loaded on device: cuda:0
Model quantized to 4-bit precision
Stopwords found.
Total Tokens Skipped: 9
Generated output Check: A) the professor will delay the midterm.


In [3]:
# LIG Method
lig_results = analyzer.analyze_layer_integrated_gradients(
    input_text=decision_prompt,
    target=generated_output,
    # static_texts=[
    #     "Choose the most plausible answer:\n",
    #     "Respond only with 'A', 'B', 'C', or 'D'.",
    # ],
    **method_params[AttributionMethod.LIG],
)
sorted_all_tokens = sorted(
    lig_results.seq_attr_dict.items(), key=lambda x: abs(x[1]), reverse=True
)[:15]
print("All tokens (sorted by absolute value):")
for token, value in sorted_all_tokens:
    print(f"{token}: {value}")

All tokens (sorted by absolute value):
Choose: 7.615722372745221
Ġdelay: 5.503748592946421
Ġprofessor: 3.9374333688965013
Ġplausible: 3.383934181558528
A: 3.321085932484744
Ġwith: 3.0709526837474783
Ġrespond: 2.7648087121973544
B: 2.265787139410205
:Ċ: 2.243938984598704
Ġonly: 2.216618759570426
Ġdescription: 2.1387169727466597
Ġmost: 1.4703437163354063
Ġmidterm: 1.45905484173938
Ġabout: 1.2967793001545251
I: 1.072513720985548


In [ ]:
# LIME Method
lime_results = analyzer.analyze_lime(
    input_text=decision_prompt,
    target=generated_output,
    # static_texts=["Choose the most plausible answer:\n",
    #                 "Respond only with 'A', 'B', 'C', or 'D'."],
    **method_params[AttributionMethod.LIME]
)
sorted_all_tokens = sorted(
    lime_results.seq_attr_dict.items(),
    key=lambda x: abs(x[1]),
    reverse=True
)[:15]
print("All tokens (sorted by absolute value):")
for token, value in sorted_all_tokens:
    print(f"{token}: {value}")

Lime attribution:   0%|          | 0/1 [00:00<?, ?it/s]

All tokens (sorted by absolute value):
Ġrespond: 1.9366235733032227
Choose: 1.9063360691070557
Ġonly: 1.8264042139053345
B: 1.6911622285842896
C: 1.3870497941970825
D: 1.2453854084014893
:Ċ: 1.1986298561096191
Ġnervous: -1.085089921951294
Ġwith: 0.9547045230865479
): 0.8225283026695251
Ġthat: -0.7677960991859436
Ġtomorrow: -0.6995872259140015
Ġfear: -0.5779537558555603
,: 0.48905980587005615
A: 0.4788435995578766
Ġprofessor: 0.4557720422744751
Ġanswer: 0.4036642611026764
.Ċ: 0.3386248052120209
.: -0.3314547836780548
Ġreceive: 0.3240361213684082
Ġit: 0.3099600672721863
Ġdiabetes: -0.301020085811615
Ġdescription: 0.28354260325431824
Ġfeeling: -0.264242947101593
Ġdood: -0.26323646306991577
Ġwill: -0.2153400331735611
Ġon: 0.2139812707901001
Ġmidterm: -0.18423153460025787
Ġmy: -0.1718088835477829
Ġand: -0.16772307455539703
ĠI: -0.15504193305969238
I: 0.1254575401544571
Ġplausible: 0.11295289546251297
Ġam: 0.09578584879636765
Ġfail: -0.0796651691198349
Ġmost: 0.06846888363361359
Ġdelay: 0.05

In [5]:
shapley_results = analyzer.analyze_shapley_value_sampling(
    input_text=decision_prompt,
    target=generated_output,
    **method_params[AttributionMethod.SHAPLEY_VALUE_SAMPLING],
)
sorted_all_tokens = sorted(
    shapley_results.seq_attr_dict.items(), key=lambda x: abs(x[1]), reverse=True
)[:15]
print("All tokens (sorted by absolute value):")
for token, value in sorted_all_tokens:
    print(f"{token}: {value}")

Shapley Value Sampling attribution:   0%|          | 0/101 [00:00<?, ?it/s]

All tokens (sorted by absolute value):
Choose: 1.873046875
Ġrespond: 1.755859375
Ġnervous: -1.1396093368530273
Ġonly: 1.1243749856948853
Ġtomorrow: -1.018593668937683
Ġabout: -0.8597655892372131
Ġfear: -0.7674218416213989
Ġprofessor: 0.7587499618530273
:Ċ: 0.6444531083106995
Ġthat: -0.619140625
): 0.5930468440055847
A: -0.5062499642372131
C: 0.4610937535762787
Ġdiabetes: -0.41398435831069946
Ġfeeling: -0.3785937428474426
